In [ ]:
import boto3
from datetime import datetime, timezone
import pandas as pd

cloudwatch = boto3.client("cloudwatch", region_name="ap-southeast-1")

# simulated data collection 
start_time = datetime(2026, 2, 27, 9, 30, 0, tzinfo=timezone.utc)
end_time   = datetime(2026, 2, 28, 2, 55, 0, tzinfo=timezone.utc)

# example ec2 instance for metrics data
instance_id = "i-05e89966d1d76016b"
host_name = "ip-172-31-44-208"


In [ ]:

response = cloudwatch.get_metric_data(
    MetricDataQueries=[
        {"Id": "cpu", "MetricStat": { "Metric": {
                                                "Namespace": "AWS/EC2",
                                                "MetricName": "CPUUtilization",
                                                "Dimensions": [
                                                    {"Name": "InstanceId", "Value": instance_id}
                                                ]
                },"Period": 300, "Stat": "Average"
                # the average over 5 minutes, as 5 minute interval is collected
            }, "ReturnData": True},

        {"Id": "network_in", "MetricStat": { "Metric": {
                                                "Namespace": "AWS/EC2",
                                                "MetricName": "NetworkIn",
                                                "Dimensions": [
                                                    {"Name": "InstanceId", "Value": instance_id}
                                                ]
                }, "Period": 300, "Stat": "Sum"
            }, "ReturnData": True },

        {"Id": "network_out", "MetricStat": { "Metric": {
                                                "Namespace": "AWS/EC2",
                                                "MetricName": "NetworkOut",
                                                "Dimensions": [
                                                    {"Name": "InstanceId", "Value": instance_id}
                                                ]
                }, "Period": 300, "Stat": "Sum"
            }, "ReturnData": True },

        {"Id": "mem_used", "MetricStat": { "Metric": {
                                                "Namespace": "CWAgent",
                                                "MetricName": "mem_used_percent",
                                                "Dimensions": [
                                                    {"Name": "host", "Value": host_name}
                                                ]
                }, "Period": 300, "Stat": "Average"
            }, "ReturnData": True 
        }

    ],
    StartTime=start_time,
    EndTime=end_time
)

In [ ]:
data = {}

for result in response["MetricDataResults"]:
    for ts, val in zip(result["Timestamps"], result["Values"]):
        if ts not in data:
            data[ts] = {}
        data[ts][result["Id"]] = val

df = pd.DataFrame.from_dict(data, orient="index")
df.sort_index(inplace=True)


In [ ]:
df = pd.DataFrame.from_dict(data, orient="index")
df.sort_index(inplace=True)

df

,cpu,network_in,network_out,mem_used
2026-02-27 17:30:00+08:00,0.479939,39328.0,34231.0,27.329994
2026-02-27 17:35:00+08:00,0.480001,38680.0,33749.0,27.327259
2026-02-27 17:40:00+08:00,0.481737,51401.0,43941.0,27.324524
2026-02-27 17:45:00+08:00,0.474933,39790.0,34579.0,27.316148
2026-02-27 17:50:00+08:00,0.476731,39034.0,33909.0,27.288454
...,...,...,...,...
2026-02-28 10:30:00+08:00,0.475069,31268.0,27351.0,27.055626
2026-02-28 10:35:00+08:00,0.476599,52271.0,45061.0,27.052891
2026-02-28 10:40:00+08:00,0.475071,38950.0,34268.0,27.048446
2026-02-28 10:45:00+08:00,0.474932,38836.0,34175.0,27.047933


In [ ]:
import boto3

# Create an EC2 client object
client = boto3.client('ec2')

response = client.describe_instances()

for reservation in response['Reservations']:
    for instance in reservation['Instances']:
        instance_id = instance['InstanceId']
        instance_state = instance['State']['Name']
        instance_type = instance['InstanceType']
        print(f"Instance ID: {instance_id}, Type: {instance_type}, State: {instance_state}")


Instance ID: i-07fab5aa324196e25, Type: t3.micro, State: terminated
